# Retail Transaction EDA Project


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Update this path if your CSV is stored elsewhere
df = pd.read_csv('retail_large_dataset.csv')
print('Shape:', df.shape)
display(df.head())
display(df.info())

## 1. Preprocessing

In [ ]:
data = df.copy()
data['order_date'] = pd.to_datetime(data['order_date'], errors='coerce')
text_cols = data.select_dtypes(include=['object']).columns
for col in text_cols:
    data[col] = data[col].astype('string').str.strip()
for col in ['gender','customer_segment','payment_method','shipping_type','return_status','product_category','product_subcategory']:
    data[col] = data[col].str.title()
print('Duplicate rows:', data.duplicated().sum())
print('Missing values by column:')
display(data.isna().sum())
# Save cleaned data
data.to_csv('retail_cleaned.csv', index=False)

## 2. Univariate analysis

In [ ]:
numeric = data.select_dtypes(include=np.number)
summary = numeric.describe().T
summary['skewness'] = numeric.skew()
summary['kurtosis'] = numeric.kurtosis()
display(summary)
summary.to_csv('numerical_summary.csv')

for col in ['product_price','quantity','discount_percentage','final_price','delivery_days','age']:
    plt.figure(figsize=(8,4))
    plt.hist(data[col].dropna(), bins=40)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col); plt.ylabel('Frequency'); plt.tight_layout(); plt.show()

for col in ['product_category','customer_segment','payment_method','return_status']:
    display(data[col].value_counts().to_frame('count'))

## 3. Outlier detection (IQR and Z-score)

In [ ]:
outlier_rows = []
for col in numeric.columns:
    q1, q3 = data[col].quantile([.25,.75])
    iqr = q3-q1
    lower, upper = q1-1.5*iqr, q3+1.5*iqr
    iqr_count = ((data[col] < lower) | (data[col] > upper)).sum()
    sd = data[col].std()
    z_count = (((data[col]-data[col].mean())/sd).abs()>3).sum() if sd else 0
    outlier_rows.append([col, lower, upper, int(iqr_count), int(z_count)])
outlier_summary = pd.DataFrame(outlier_rows, columns=['variable','IQR_lower','IQR_upper','IQR_outlier_count','Zscore_abs_gt_3_count'])
display(outlier_summary)
outlier_summary.to_csv('outlier_summary.csv', index=False)
# Keep potential valid high-value transactions unless verified as errors.

## 4. Bivariate analysis and correlation

In [ ]:
corr = numeric.corr()
plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Numerical Correlation Heatmap'); plt.tight_layout(); plt.show()
display(corr)
print('Category vs return rate (%)')
display(data.groupby('product_category').return_status.apply(lambda s:(s=='Yes').mean()*100).sort_values(ascending=False))
print('Customer segment average final price')
display(data.groupby('customer_segment').final_price.mean().sort_values(ascending=False))
print('Shipping type delivery days')
display(data.groupby('shipping_type').delivery_days.agg(['mean','median','count']))

## 5. Multivariate analysis

In [ ]:
# Category × discount × revenue
display(data.groupby(['product_category', pd.cut(data.discount_percentage, [0,10,20,30], include_lowest=True)]).final_price.agg(['count','mean','sum']))
# Segment × age band × spending
data['age_group'] = pd.cut(data.age, bins=[0,29,45,60,120], labels=['Under 30','30–45','46–60','Over 60'])
display(data.groupby(['customer_segment','age_group'], observed=False).final_price.agg(['count','mean','sum']))
# Shipping × delivery duration × return status
display(data.groupby(['shipping_type','return_status']).delivery_days.agg(['count','mean','median']))

## 6. Business summary and exports

In [ ]:
print('Total transactions:', len(data))
print('Total final-price value:', data.final_price.sum())
print('Average final price:', data.final_price.mean())
print('Overall return rate (%):', (data.return_status=='Yes').mean()*100)
category_summary = data.groupby('product_category').agg(
    transactions=('order_id','count'),
    revenue=('final_price','sum'),
    average_order_value=('final_price','mean'),
    return_rate=('return_status',lambda x:(x=='Yes').mean()*100)
).sort_values('revenue',ascending=False)
display(category_summary)
category_summary.to_csv('category_summary.csv')